# Lab 3.2: Compaction and Handoff Verifier

**Day 1 - Topic 3**

Goal: audit an unfocused auto-compaction summary, a focused /compact summary, and a HANDOFF.md against what git status and pytest report for the repository.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab3_2_compaction_and_handoff_verifier/start/lab3_2_compaction_and_handoff_verifier.ipynb)

In [ ]:
# Run this cell once to install required packages.
# If running locally with the course venv already activated, you can skip this.
!pip install "pytest>=7" --quiet
# This lab also uses git, which Colab provides.

In [ ]:
import ast
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/context-claude-companion.git"


def find_repo_root() -> Path:
    """Use the local companion checkout if the notebook runs inside it; otherwise clone it (Colab)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "day1" / "demos" / "fixtures").is_dir():
            return p
    clone = Path("context-claude-companion").resolve()
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return clone


REPO_ROOT = find_repo_root()
FIXTURES = REPO_ROOT / "day1" / "demos" / "fixtures"
print(f"Using fixtures in {FIXTURES}")


def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: The mid-session repo

`compaction_repo` is a refund-ledger bug (separate from Lab 3.1). The helper copies it to a temporary folder, commits a baseline, and applies one in-progress edit to `refunds/ledger.py`, as if you stopped mid-session. `session_history/` holds three summaries of that session. Here is what automatic compaction produced.

In [ ]:
FIXTURE = FIXTURES / "compaction_repo"
GIT = ["git", "-c", "user.name=demo", "-c", "user.email=demo@example.com"]


def sh(root: Path, cmd: list[str]) -> str:
    proc = subprocess.run(cmd, cwd=root, capture_output=True, text=True)
    return proc.stdout + proc.stderr


def build_session_state(root: Path) -> None:
    """Commit the fixture, then leave the working tree as it was mid-session."""
    subprocess.run(["git", "init", "-q"], cwd=root, check=True)
    subprocess.run(["git", "add", "-A"], cwd=root, check=True)
    subprocess.run(GIT + ["commit", "-qm", "baseline"], cwd=root, check=True)
    ledger = root / "refunds/ledger.py"
    ledger.write_text(ledger.read_text().replace(
        "        self._seen: set[str] = set()\n",
        "        self._seen: set[str] = set()\n        self._seen_at: dict[str, int] = {}  # WIP: first-seen order\n",
    ))

In [ ]:
print((FIXTURE / "session_history" / "auto_compact_summary.md").read_text())

## Section 2: Collect ground truth

A summary is only trustworthy if it matches the repository. Gather the facts from tools: modified files from `git status --porcelain`, failing tests from `pytest`, and the method signature from `ast`. The rejected approaches come from the session transcript and are provided.

In [ ]:
def collect_ground_truth(root: Path) -> dict:
    status = sh(root, ["git", "status", "--porcelain"]).rstrip().splitlines()
    # TODO: Build modified: the file path of every status line that does NOT start with "??"  (the path starts at column 3, so use line[3:])
    modified: list[str] = []

    pytest_out = sh(root, [sys.executable, "-m", "pytest", "-q", "-rf", "--tb=no", "--color=no", "-p", "no:cacheprovider"])
    # TODO: Build failing: re.findall(r"^FAILED (\S+)", pytest_out, flags=re.M) returns the failing test ids
    failing: list[str] = []

    tree = ast.parse((root / "refunds/ledger.py").read_text())
    # TODO: Find the ast.FunctionDef named "post_refund" among ast.walk(tree)  (hint: next(n for n in ast.walk(tree) if ...))
    # TODO: Build signature as f"post_refund({ast.unparse(func.args)}) -> {ast.unparse(func.returns)}"
    signature = ""

    transcript = (root / "session_history/transcript_excerpt.md").read_text()
    rejected = re.findall(r"`(\w+)`[^\n]*REJECTED", transcript)

    return {"status": status, "modified": modified, "pytest_out": pytest_out,
            "failing": failing, "signature": signature, "rejected": rejected}

In [ ]:
workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name).resolve() / "compaction_repo"
shutil.copytree(FIXTURE, root)
build_session_state(root)
truth = collect_ground_truth(root)

print("=== Ground truth from the repository ===")
print(f"  git status --porcelain : {', '.join(l.strip() for l in truth['status']) or '(clean)'}")
print(f"  pytest failing         : {', '.join(truth['failing']) or '(none)'}")
print(f"  signature (ast)        : {truth['signature']}")
print(f"  rejected (transcript)  : {', '.join(truth['rejected'])}")

# Expected:
#   git status --porcelain : M refunds/ledger.py
#   pytest failing         : tests/test_ledger.py::test_duplicate_delivery_is_not_posted_twice
#   signature (ast)        : post_refund(self, order_id: str, amount_cents: int, idempotency_key: str) -> bool
#   rejected (transcript)  : ttl_dedupe_cache, global_post_lock
print()
check("Only refunds/ledger.py is modified", truth["modified"] == ["refunds/ledger.py"])
check("One test fails: test_duplicate_delivery_is_not_posted_twice",
      truth["failing"] == ["tests/test_ledger.py::test_duplicate_delivery_is_not_posted_twice"])
check("The ast signature of post_refund is extracted",
      truth["signature"] == "post_refund(self, order_id: str, amount_cents: int, idempotency_key: str) -> bool")

## Section 3: Audit each summary

`audit()` is provided. For one summary it checks four things: does it name the failing test, how many rejected fixes does it keep, does it contain the real signature, and does it name an unmodified file as work in progress.

In [ ]:
def audit(text: str, truth: dict) -> dict:
    """Check one summary against the facts that git, pytest, ast, and the transcript produced."""
    named_as_work = set()
    for line in text.splitlines():
        if re.search(r"edit|modif|fix|chang|updat", line, flags=re.I):
            named_as_work.update(re.findall(r"[\w/]+\.py", line))
    named_as_work = {p for p in named_as_work if not p.startswith("tests/")}
    return {
        "failing_test": all(f in text for f in truth["failing"]),
        "rejected": sum(1 for r in truth["rejected"] if r in text),
        "signature": truth["signature"] in text,
        "unmodified_named": sorted(named_as_work - set(truth["modified"])),
    }

## Section 4: Compare the three artifacts

Audit the automatic summary, the focused `/compact` summary, and `HANDOFF.md` against the ground truth you collected.

In [ ]:
artifacts = [
    ("1. Unfocused auto-compaction", "auto_compact_summary.md"),
    ("2. Focused /compact summary", "focused_compact_summary.md"),
    ("3. HANDOFF.md", "HANDOFF.md"),
]
results = {}
print(f"{'Artifact':<30} | {'Failing test':<12} | {'Rejected fixes':<14} | {'Signature':<9} | Unmodified files named as work")
print("-" * 110)
for label, name in artifacts:
    r = audit((root / "session_history" / name).read_text(), truth)
    results[name] = r
    rejected_ratio = f"{r['rejected']}/{len(truth['rejected'])}"
    print(f"{label:<30} | {str(r['failing_test']):<12} | {rejected_ratio:<14} | "
          f"{str(r['signature']):<9} | {', '.join(r['unmodified_named']) or 'none'}")
workdir.cleanup()

# Expected:
# 1. Unfocused auto-compaction   | False        | 0/2            | False     | refunds/webhook.py
# 2. Focused /compact summary    | True         | 2/2            | True      | none
# 3. HANDOFF.md                  | True         | 2/2            | True      | none
print()
auto = results["auto_compact_summary.md"]
good = {"failing_test": True, "rejected": 2, "signature": True, "unmodified_named": []}
check("Auto-compaction dropped the failing test, both rejected fixes, and the signature",
      auto["failing_test"] is False and auto["rejected"] == 0 and auto["signature"] is False)
check("Auto-compaction points the next step at an unmodified file (refunds/webhook.py)",
      auto["unmodified_named"] == ["refunds/webhook.py"])
check("The focused /compact summary keeps every fact", results["focused_compact_summary.md"] == good)
check("HANDOFF.md keeps every fact", results["HANDOFF.md"] == good)

In [ ]:
print("Lab complete.")
print("Takeaway: An automatic summary can keep the story and drop the failing test, the rejected fixes, and the signature. Check any summary or HANDOFF.md against git status and pytest before you resume work.")